### Test für Batching in ESMPSeudoperplexität:

In [ ]:
import time
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR
from aguirre.evaluation import ESMPseudoPerplexityScorer

cfg = Config()
sc = ESMPseudoPerplexityScorer(cfg)

seqs = (
    pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"]
    .head(40).tolist()
)
print(f"{len(seqs)} Sequenzen, Laengen {min(map(len, seqs))}-{max(map(len, seqs))}")

t = time.perf_counter()
a = sc._score_schleife(seqs)["pppl_esm2"].to_numpy()
t_schleife = time.perf_counter() - t

t = time.perf_counter()
b = sc._score_batch(seqs)["pppl_esm2"].to_numpy()
t_batch = time.perf_counter() - t

print(f"Schleife {t_schleife:.1f}s | Batch {t_batch:.1f}s | Faktor {t_schleife / t_batch:.1f}")
print("max. abs. Abweichung:", float(np.abs(a - b).max()))
assert np.allclose(a, b, rtol=1e-5)
print("Im Rahmen der Gleitkommagenauigkeit identisch.")

### Test für Batching in ProGen2 Perplexität:

In [ ]:
import time
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR
from aguirre.evaluation import ProGen2Scorer

cfg = Config()
sc = ProGen2Scorer(cfg)

seqs = (
    pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"]
    .head(40).tolist()
)
print(f"{len(seqs)} Sequenzen, Laengen {min(map(len, seqs))}-{max(map(len, seqs))}")

t = time.perf_counter()
a = sc._score_schleife(seqs)["ppl_progen2"].to_numpy()
t_schleife = time.perf_counter() - t

t = time.perf_counter()
b = sc._score_batch(seqs)["ppl_progen2"].to_numpy()
t_batch = time.perf_counter() - t

print(f"Schleife {t_schleife:.1f}s | Batch {t_batch:.1f}s | Faktor {t_schleife / t_batch:.1f}")
print("max. rel. Abweichung:", float((np.abs(a - b) / a).max()))
assert np.allclose(a, b, rtol=1e-4)

In [ ]:
seqs = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"].head(200).tolist()
for bs in (8, 16, 32, 64, 128):
    sc.batch_size = bs
    t = time.perf_counter(); sc._score_batch(seqs); dt = time.perf_counter() - t
    print(f"batch_size {bs:4d}: {dt:5.1f}s  ->  {dt/len(seqs)*1000:5.1f} ms/Sequenz")

In [ ]:
kurz = seqs[int(np.argmin([len(s) for s in seqs]))]
lang = seqs[int(np.argmax([len(s) for s in seqs]))]

allein = sc._score_batch([kurz])["ppl_progen2"].iloc[0]
zusammen = sc._score_batch([kurz, lang])["ppl_progen2"].iloc[0]
print(f"allein {allein:.6f} | neben {len(lang)}er {zusammen:.6f} | "
      f"rel. Diff {abs(allein-zusammen)/allein:.2e}")
assert np.isclose(allein, zusammen, rtol=1e-4)

### Vorbereitung für KnownAMPSimilirityScorer

In [ ]:
from aguirre.data import write_fasta
from aguirre.config import TRACKED_DIR
import pandas as pd
PATH = TRACKED_DIR / "torres_train_sequences.csv"
write_fasta(pd.read_csv(PATH)["sequence"].tolist(), TRACKED_DIR / "torres_train.fasta")

In [ ]:
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR, PROJECT_ROOT
from aguirre.data import write_fasta
from aguirre.evaluation import (
    SequenceSet, KnownAMPSimilarityScorer,
    DistributionComparison, CompositionComparison,
)

cfg = Config()
WOG = PROJECT_ROOT / "wrath_of_god"

# ---- Daten ----------------------------------------------------------------
train_df = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")
gen_df = pd.read_csv(WOG / "results" / "50k_samples" / "ampdiffusion_650M_50k.csv")
print(f"train {len(train_df)} | generiert {len(gen_df)}")

if not cfg.amp_referenz_fasta.exists():
    write_fasta(train_df["sequence"].tolist(), cfg.amp_referenz_fasta)

echte = train_df["sequence"].head(30).tolist()
generierte = gen_df["sequence"].head(30).tolist()

# ---- 1. Kontrollen mit bekanntem Ausgang ----------------------------------
sc = KnownAMPSimilarityScorer.from_config(cfg)
ref = train_df["sequence"].tolist()
kontrollen = [ref[0], ref[1][: len(ref[1]) // 2], "WWWWWWWWWWWW"]
k = sc.score(kontrollen)[sc.columns[0]].to_numpy()
print(f"\nidentisch {k[0]:.3f} | halbiert {k[1]:.3f} | Unsinn {k[2]:.3f}")
assert k[2] == 0.0, "Unsinn muss 0.0 geben, nicht NaN -- fill_value fehlt"

# ---- 2. 30 echte + 30 generierte, gemischt --------------------------------
gemischt = echte + generierte
df = sc.score(gemischt)
df["art"] = ["train"] * 30 + ["generiert"] * 30
assert len(df) == 60, "Zeilenzahl falsch -- Rueckzuordnung kaputt"
assert np.allclose(df.loc[df["art"] == "train", "max_ident_bekannt"], 1.0), \
    "Trainingssequenzen muessen sich selbst mit 1.0 finden"

print("\n", df.groupby("art")["max_ident_bekannt"].describe()[["mean", "50%", "max"]])
ueber = (df.loc[df["art"] == "generiert", "max_ident_bekannt"] > cfg.similarity_max_ident)
print(f"generierte ueber Torres' Schwelle {cfg.similarity_max_ident}: {ueber.sum()}/30")

# ---- 3. Laengen und Komposition, ohne jeden Scorer ------------------------
gruppen = {
    "train":     SequenceSet("train", pd.DataFrame({"sequence": echte})),
    "generiert": SequenceSet("generiert", pd.DataFrame({"sequence": generierte})),
}
OUT = WOG / "runs" / "smoke_eval"
DistributionComparison("length").compare(gruppen).save(OUT)
CompositionComparison(reference="train").compare(gruppen).save(OUT)
print("\n", sorted(p.name for p in OUT.iterdir()))

### Test für identitätswert:

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR, PROJECT_ROOT, TMP_DIR
from aguirre.data import write_fasta, run_mmseqs_search, M8_SPALTEN
from aguirre.evaluation import KnownAMPSimilarityScorer

cfg = Config()
WOG = PROJECT_ROOT / "wrath_of_god"

train_df = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")
gen_df = pd.read_csv(WOG / "results" / "50k_samples" / "ampdiffusion_650M_50k.csv")
ref = train_df["sequence"].tolist()
generierte = gen_df["sequence"].head(30).tolist()

# ---- 1. Kommen die Parameter wirklich aus der Config an? -------------------
sc = KnownAMPSimilarityScorer.from_config(cfg)
print(f"cov_mode={sc.cov_mode}  coverage={sc.coverage}  evalue={sc.evalue:g}")
print("Fingerabdruck:", sc.fingerprint)
assert (sc.cov_mode, sc.coverage) == (cfg.similarity_cov_mode, cfg.similarity_coverage)

# ---- 2. Drei Kontrollen ----------------------------------------------------
kontrollen = [ref[0], ref[1][: len(ref[1]) // 2], "WWWWWWWWWWWW"]
k = sc.score(kontrollen)[sc.columns[0]].to_numpy()
print(f"\nidentisch {k[0]:.3f} | halbiert {k[1]:.3f} | Unsinn {k[2]:.3f}")
assert np.isclose(k[0], 1.0), "Eine Trainingssequenz muss sich selbst mit 1.0 finden"
assert k[2] == 0.0, "Unsinn muss 0.0 sein"

# ---- 3. Die 30 generierten: streuen sie jetzt? -----------------------------
w = sc.score(generierte)[sc.columns[0]].to_numpy()
print("\nmax_ident_bekannt der 30 generierten:")
print(pd.Series(w).describe()[["mean", "50%", "min", "max"]].round(3).to_dict())
print("Werte sortiert:", np.round(np.sort(w)[::-1], 3).tolist())
print(f"ueber Torres' Schwelle {cfg.similarity_max_ident}: {(w > cfg.similarity_max_ident).sum()}/30")
assert len(np.unique(np.round(w, 3))) > 1, "Immer noch keine Streuung -- Gate greift nicht"

# ---- 4. Beissen die neuen Parameter? Rohtreffer ansehen --------------------
with tempfile.TemporaryDirectory(dir=TMP_DIR) as tmp:
    tmp = Path(tmp)
    q = write_fasta(generierte, tmp / "q.fasta")
    out = run_mmseqs_search(
        query_fasta_path=q, out_path=tmp / "h.m8", tmp_dir=tmp,
        target_fasta_path=cfg.amp_referenz_fasta,
        cov_mode=cfg.similarity_cov_mode,
        coverage_rate=cfg.similarity_coverage,
        evalue=cfg.similarity_evalue,
    )
    hits = (pd.read_csv(out, sep="\t", names=M8_SPALTEN)
            if Path(out).stat().st_size else pd.DataFrame(columns=M8_SPALTEN))

if len(hits):
    print(f"\nTreffer gesamt: {len(hits)} | je Query im Mittel: "
          f"{hits.groupby('query').size().mean():.1f}  (vorher: 513)")
    print("alnlen/qlen:", (hits['alnlen'] / hits['qlen']).describe()[['mean','min']].round(3).to_dict())
    print("qcov min:", round(float(hits['qcov'].min()), 3), f" (muss >= {cfg.similarity_coverage})")
else:
    print("\nKeine Treffer -- alle 30 sind von allem Bekannten verschieden.")

In [ ]:
alle = gen_df["sequence"].tolist()
w = sc.score(alle)["max_ident_bekannt"].to_numpy()

print(f"n = {len(w)}")
print(f"ueberhaupt ein Treffer : {(w > 0).mean():.1%}")
print(f"ueber {cfg.similarity_max_ident}      : {(w > cfg.similarity_max_ident).mean():.2%}")
print(f"exakte Kopien (= 1.0)  : {int((w >= 0.999).sum())}")
print("\nbedingt auf Treffer:", pd.Series(w[w > 0]).describe()[["count","mean","50%"]].round(3).to_dict())

### Smoke-Test für Diversität

In [1]:
"""Test des DiversityFilter -- in ein Notebook oder als scripts/test_diversity.py."""

import numpy as np
import pandas as pd
from aguirre.config import Config, TRACKED_DIR
from aguirre.evaluation import SequenceSet, DiversityFilter

cfg = Config()
train = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"].tolist()
df = DiversityFilter.from_config(cfg)

def mache_set(seqs, mics, name="test"):
    return SequenceSet(name, pd.DataFrame({"sequence": seqs, "mic_apex": mics}))


# ---- 0. Zwei wirklich unaehnliche Sequenzen finden ------------------------
# Nicht raten: zwei zufaellig gewaehlte AMPs koennen einander aehneln, und
# dann schlaegt der eigentliche Test aus dem falschen Grund fehl.
a = train[0]
b = None
for kandidat in train[1:60]:
    probe = df.apply(mache_set([a, kandidat], [1.0, 2.0], "probe"))
    if len(probe.data) == 2:
        b = kandidat
        break
assert b is not None, "Keine unaehnliche Partnersequenz gefunden -- Suchbereich erweitern."
print(f"unaehnliches Paar gefunden: len {len(a)} und {len(b)}")


# ---- 1. Der eigentliche Test: 3x identisch + 1 verschieden ---------------
# mic_apex so gewaehlt, dass der Gewinner eindeutig ist: Index 1 (10.0).
ss = mache_set([a, a, a, b], [50.0, 10.0, 90.0, 30.0], "drei_plus_eins")
raus = df.apply(ss)

print("\nbehalten:")
print(raus.data[["sequence", "mic_apex"]].assign(
    sequence=lambda d: d["sequence"].str[:18] + "..."))

assert len(raus.data) == 2, f"Erwartet 2 Sequenzen, bekam {len(raus.data)}"
behaltene_a = raus.data.loc[raus.data["sequence"] == a, "mic_apex"].tolist()
assert behaltene_a == [10.0], (
    f"Von den drei Identischen muss die mit MIC 10.0 uebrig bleiben, "
    f"uebrig blieb {behaltene_a}"
)
assert b in raus.data["sequence"].tolist(), "Die unaehnliche Sequenz fehlt"
print("1 drei identische + eine andere    ok")


# ---- 2. Alles verschieden -> nichts wird verworfen -----------------------
verschiedene = [a, b]
ss2 = mache_set(verschiedene, [1.0, 2.0], "verschieden")
assert len(df.apply(ss2).data) == 2
print("2 unaehnliche bleiben unangetastet ok")


# ---- 3. Determinismus ----------------------------------------------------
r1 = df.apply(ss).data["sequence"].tolist()
r2 = df.apply(ss).data["sequence"].tolist()
assert r1 == r2, "Zwei Laeufe, zwei Ergebnisse -- die Sortierung greift nicht"
print("3 deterministisch                  ok")


# ---- 4. Ohne mic_apex: Eingabereihenfolge entscheidet --------------------
# Dann gewinnt die ERSTE der drei Identischen, nicht die beste.
ohne = SequenceSet("ohne_mic", pd.DataFrame({"sequence": [a, a, a, b]}))
raus4 = df.apply(ohne)
assert len(raus4.data) == 2
print("4 Rueckfall ohne Sortierspalte     ok  (Warnung oben muss erschienen sein)")

print("\nAlle Tests gruen.")

[DiversityFilter] 2 von 2 behalten (100.0%), Schwelle 0.4.
unaehnliches Paar gefunden: len 13 und 25
[DiversityFilter] 2 von 4 behalten (50.0%), Schwelle 0.4.

behalten:
                sequence  mic_apex
0       FFKKWPWWPWRRK...      10.0
1  RSVCRQIKICRRRGGCYY...      30.0
1 drei identische + eine andere    ok
[DiversityFilter] 2 von 2 behalten (100.0%), Schwelle 0.4.
2 unaehnliche bleiben unangetastet ok
[DiversityFilter] 2 von 4 behalten (50.0%), Schwelle 0.4.
[DiversityFilter] 2 von 4 behalten (50.0%), Schwelle 0.4.
3 deterministisch                  ok
[DiversityFilter] Spalte 'mic_apex' fehlt -- Eingabereihenfolge entscheidet die Konflikte.
[DiversityFilter] 2 von 4 behalten (50.0%), Schwelle 0.4.
4 Rueckfall ohne Sortierspalte     ok  (Warnung oben muss erschienen sein)

Alle Tests gruen.
